In [ ]:
import deeplabcut as dlc
dlc.__version__

In [ ]:
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from pathlib import Path

In [ ]:
PROJECT_NAME: str = "Stick_Insect_Motion_Hacking_2D_Tracking"
EXPERIMENTER: str = "RrPllana"
VIDEO_DIR: str = "videos/" 
WORKING_DIR: str = "models/"
# Dynamically create a list of video paths in the VIDEO_DIR that match the pattern "PIC_*.mp4"
video_paths = [str(path) for path in Path(VIDEO_DIR).glob("PIC_*.mp4")]

path_config_file = dlc.create_new_project(PROJECT_NAME, EXPERIMENTER, video_paths, copy_videos=True)

In [ ]:
CONFIG_PATH: str = path_config_file
print(f"Configuration file created at: {CONFIG_PATH}")

In [ ]:
import yaml

with open(CONFIG_PATH, 'r') as file:
    config = yaml.safe_load(file)

# Update the config dictionary with the new values from the user input without editing the config.yaml file directly
config['numframes2pick'] = 22  # Update the number of frames to pick for labeling/refinement

# Check if the 'bodyparts' and 'skeleton' keys are presented correctly in the config dictionary
print("Number of frames to pick for labeling/refinement:", config['numframes2pick'])
print("Bodyparts:", config['bodyparts'])
print("Skeleton:", config['skeleton'])

In [ ]:
%matplotlib inline
dlc.extract_frames(CONFIG_PATH, mode='automatic', userfeedback=False, crop=False)

In [ ]:
%gui qt6
import napari

# If this is used, then first after opening the view, it is needed to go to Plugins -> napari-deeplabcut, to be able to proces multiple labels
# and also save the files correctly, otherwise it will not work properly.

# Next you drag and drop config file and the folder that contains the extracted frames you want to label.
napari.Viewer()

In [ ]:
# Alternative to napari.Viewer() command, it can be used to open the view with the config file and the folder that contains the extracted 
# frames you want to label.
dlc.label_frames(CONFIG_PATH)

In [ ]:
dlc.check_labels(CONFIG_PATH, draw_skeleton=True)

In [ ]:
dlc.create_training_dataset(CONFIG_PATH)

In [ ]:
# Running with GPU and specifying the network type as "resnet_50" and engine as "pytorch"
dlc.create_training_dataset(CONFIG_PATH, net_type="resnet_50", engine=dlc.Engine.PYTORCH)

In [ ]:
dlc.train_network(
    CONFIG_PATH,   # path to the config.yaml file
    shuffle=1,     # shuffle the training data
    save_epochs=5, # save the model every 5 epochs
    epochs=200,    # number of training epochs
    batch_size=8,  # batch size for training (changing this number changes how we utilize the GPU memory)
    device=device,  # specify the device to use (GPU or CPU)
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

In [ ]:
dlc.analyze_videos(CONFIG_PATH, videos=video_paths, videotype='mp4', 
                   shuffle=1, trainingsetindex=0, gputouse=None, save_as_csv=False,
                   destfolder=VIDEO_DIR, batchsize=1, cropping=None,
                   dynamic=(False, 0.5, 10))

In [ ]:
dlc.extract_outlier_frames(CONFIG_PATH, video_paths)  # pass a specific video

In [ ]:
%gui qt6
dlc.refine_labels(CONFIG_PATH)

In [ ]:
dlc.create_labeled_video(CONFIG_PATH, video_paths,  videotype='.mp4', destfolder=VIDEO_DIR)

In [ ]:
%matplotlib notebook
dlc.plot_trajectories(CONFIG_PATH, video_paths)